# RandomizedSearchCV (HistGradientBoosting)
**tabular › tuning_classification** · notebook `randomized_search_hgb`

**Kapan dipakai:** Tuning tanpa Optuna, hanya sklearn. Pencarian acak biasanya jauh lebih efisien daripada grid untuk >3 hiperparameter.

**Jebakan umum:** Nama parameter harus diawali 'model__' karena model ada di dalam Pipeline.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular/classification/titanic_train.csv"
TEST_PATH  = ROOT / "data_latihan/tabular/classification/titanic_test.csv"      # None kalau tidak ada file test
TARGET     = "survived"
ID_COL     = 'id'
DROP_COLS  = ['alive']
GROUP_COL  = None                      # isi bila ada grup agar tidak bocor
SCORING    = "f1_macro"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "randomized_search_hgb"
TASK       = "classification"
KIND       = "tree"
OUT = out_dir("tabular", "tuning_classification", MODEL_KEY)
set_seed(SEED)

## 2. Baca data

In [ ]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH else None
print(df.shape, None if test is None else test.shape)
display(df.head(3))

## 3. Fitur & split

In [ ]:
from superdb.tabular import detect_columns, prepare_features, build_pipeline, split_train_valid, cv_score
cols = detect_columns(df, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS)
X, NUM, CAT = prepare_features(df, cols)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df[TARGET])
print("kelas:", dict(enumerate(le.classes_)))
groups = df[GROUP_COL] if GROUP_COL else None
Xtr, Xva, ytr, yva = split_train_valid(X, y, TASK, VALID_SIZE, SEED, groups)
GTR = None if groups is None else groups.iloc[Xtr.index]   # grup untuk CV
print(Xtr.shape, Xva.shape)

## 4. Model dasar & ruang pencarian
Nama parameter diawali `model__` karena model berada di dalam Pipeline.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from scipy.stats import loguniform, randint

base = HistGradientBoostingClassifier(random_state=42)
param_dist = {
    "model__learning_rate": loguniform(0.01, 0.3),
    "model__max_iter": randint(100, 600),
    "model__max_leaf_nodes": randint(8, 64),
    "model__min_samples_leaf": randint(5, 60),
    "model__l2_regularization": loguniform(1e-3, 10),
}
SEARCH = "random"   # "random" atau "grid"

## 5. Jalankan pencarian

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, StratifiedKFold, KFold
pipe0 = build_pipeline(base, NUM, CAT, kind=KIND)
cv = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED) if TASK == "classification" else KFold(N_SPLITS, shuffle=True, random_state=SEED)
if SEARCH == "random":
    search = RandomizedSearchCV(pipe0, param_dist, n_iter=20, scoring=SCORING, cv=cv, random_state=SEED, n_jobs=-1)
else:
    search = GridSearchCV(pipe0, param_dist, scoring=SCORING, cv=cv, n_jobs=-1)
search.fit(Xtr, ytr)
print("skor CV terbaik:", round(search.best_score_, 4)); print(search.best_params_)
res = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
display(res[["rank_test_score", "mean_test_score", "std_test_score", "params"]].head(8))

## 6. Evaluasi pada validation

In [ ]:
pipe = search.best_estimator_

In [ ]:
pred = pipe.predict(Xva)
proba = pipe.predict_proba(Xva)
metrics = classification_metrics(yva, pred, proba, classes=np.arange(proba.shape[1]))
display(show_table(metrics))

In [ ]:
save_json({"best_cv": search.best_score_, "best_params": search.best_params_, "valid": metrics}, OUT / "metrics.json")

## Latih ulang pada semua data → submission → simpan

In [ ]:
final = __import__('sklearn.base', fromlist=['clone']).clone(search.best_estimator_).fit(X, y)
if test is not None:
    Xte, _, _ = prepare_features(test, cols)
    Xte = Xte.reindex(columns=X.columns)
    ids = test[ID_COL] if ID_COL else np.arange(len(test))
    sub = make_submission(ids, le.inverse_transform(final.predict(Xte)), ID_COL or "id", TARGET, OUT / "submission.csv")
    display(sub.head(3))
save_model({"pipe": final, "cols": cols}, OUT / "model.joblib")
reload_pipe = load_model(OUT / "model.joblib")["pipe"]
assert np.allclose(reload_pipe.predict(X.head(50)), final.predict(X.head(50))), "model muat-ulang beda"
print("OK. hasil di", OUT)